In [2]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# ── Load the 3 annotated files ──
a1 = pd.read_excel('/content/annotation_subset_200anni (2).xlsx')    # change names
a2 = pd.read_excel('/content/annotation_subset_200manoshi.xlsx')
a3 = pd.read_excel('/content/sadmanscoring.xlsx')

cats = ['cultural','political','sports','religious','health','celebrity','international']

# ── Clean: make sure category columns are numeric 0/1 ──
for df_ann in [a1, a2, a3]:
    for cat in cats:
        df_ann[cat] = pd.to_numeric(df_ann[cat], errors='coerce').fillna(0).astype(int)

# ── Pairwise Cohen's Kappa per category ──
print("=" * 65)
print(f"{'Category':15s}  {'A1-A2':8s}  {'A1-A3':8s}  {'A2-A3':8s}  {'Mean κ':8s}")
print("=" * 65)

overall_kappas = []
kappa_table = []   # for paper table

for cat in cats:
    v1 = a1[cat].values
    v2 = a2[cat].values
    v3 = a3[cat].values

    k12 = cohen_kappa_score(v1, v2)
    k13 = cohen_kappa_score(v1, v3)
    k23 = cohen_kappa_score(v2, v3)
    mean_k = (k12 + k13 + k23) / 3
    overall_kappas.append(mean_k)
    kappa_table.append({'category': cat, 'A1_A2': round(k12,3),
                        'A1_A3': round(k13,3), 'A2_A3': round(k23,3),
                        'mean_kappa': round(mean_k,3)})
    print(f"  {cat:13s}  {k12:6.3f}    {k13:6.3f}    {k23:6.3f}    {mean_k:6.3f}")

print("=" * 65)
macro_kappa = sum(overall_kappas) / len(overall_kappas)
print(f"  {'Macro mean':13s}                                  {macro_kappa:6.3f}")
print()
print("Kappa guide: <0.40 poor | 0.40-0.60 moderate | 0.61-0.80 good | >0.80 excellent")

# ── Save kappa table for paper ──
kappa_df = pd.DataFrame(kappa_table)
kappa_df.to_csv('iaa_kappa_results.csv', index=False)
print("\nSaved: iaa_kappa_results.csv  ← paste this into your paper as Table X")

# ── Majority vote: create gold-standard labels from 3 annotators ──
print("\n=== Creating gold-standard labels by majority vote ===")
gold = a1[['item_id','text','label','source']].copy()
for cat in cats:
    votes = a1[cat].values + a2[cat].values + a3[cat].values
    gold[cat] = (votes >= 2).astype(int)   # majority = 2 or 3 out of 3

gold.to_excel('annotation_200_goldstandard.xlsx', index=False)
print("Saved: annotation_200_goldstandard.xlsx  ← this is your clean test subset")
print(f"\nGold label distribution:")
for cat in cats:
    print(f"  {cat:15s}: {gold[cat].sum():3d} / 200")

Category         A1-A2     A1-A3     A2-A3     Mean κ  
  cultural        1.000     0.277     0.277     0.518
  political       1.000     0.519     0.519     0.679
  sports          1.000     0.771     0.771     0.847
  religious       1.000     0.651     0.651     0.767
  health          1.000     0.792     0.792     0.861
  celebrity       1.000     0.740     0.740     0.827
  international   1.000     0.582     0.582     0.721
  Macro mean                                      0.746

Kappa guide: <0.40 poor | 0.40-0.60 moderate | 0.61-0.80 good | >0.80 excellent

Saved: iaa_kappa_results.csv  ← paste this into your paper as Table X

=== Creating gold-standard labels by majority vote ===
Saved: annotation_200_goldstandard.xlsx  ← this is your clean test subset

Gold label distribution:
  cultural       :  13 / 200
  political      :  58 / 200
  sports         :  32 / 200
  religious      :  10 / 200
  health         :  20 / 200
  celebrity      :  30 / 200
  international  :  40 / 200